Used to load large h5ad objects in chunks and slice them to remove unneeded genes, allowing pseudobulk processing of initially large (>30GB) h5ad objects.
* Generates the SEA-AD_REGION_Processed.h5ad files used in the figures.ipynb analysis

In [ ]:
import os
#pip install ipykernel, anndata, pandas.
print("Current Working Directory:", os.getcwd())
os.chdir("../")
#Verify current directory and list the files
print("Files in this directory:", os.listdir())

In [ ]:
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import gc
import time
import anndata as ad
import anndata._io.specs, anndata._io.h5ad
import scanpy
import h5py
import numpy as np
import scipy.sparse as sp
from src.plot_utils import assign_APJ_Cats

# CONFIGURATION
regions=['HIP','DFC','MTG']
for region in regions:
    INPUT_FILE = f"large_data/SEAAD_{region}_RNAseq_final-nuclei.2026-06-22.h5ad"
    OUTPUT_FILE = f"large_data/SEA-AD_{region}_Processed.h5ad"


    TARGET_GENES = ["APLN", "APLNR", "RBFOX3", "GFAP", "AIF1"]
    CHUNK_SIZE = 10000  # 10k cells per batch


    #Adata opening in backed mode. 
    print(f"Opening {INPUT_FILE} in backed mode...")
    orig_read_elem = anndata._io.specs.read_elem

    def safe_read_elem(elem, *args, **kwargs):
        if getattr(elem, "name", "").endswith("/layers"):
            return {}  # Skip reading the UMI matrix into memory
        return orig_read_elem(elem, *args, **kwargs)

    anndata._io.specs.read_elem = safe_read_elem
    anndata._io.h5ad.read_elem = safe_read_elem

    adata_backed = ad.read_h5ad(INPUT_FILE, backed="r")

    #Restore original helper so nothing else is affected
    anndata._io.specs.read_elem = orig_read_elem
    anndata._io.h5ad.read_elem = orig_read_elem

    # Match available genes
    valid_genes = [g for g in TARGET_GENES if g in adata_backed.var_names]
    missing_genes = set(TARGET_GENES) - set(valid_genes)
    if missing_genes:
        print(f"Warning: Missing genes: {missing_genes}")

    gene_indices = [adata_backed.var_names.get_loc(g) for g in valid_genes]
    total_cells = adata_backed.n_obs
    print(f"Total cells to process: {total_cells:,}")
    print(f"Target genes ({len(valid_genes)}): {valid_genes}\n")


    #open h5py pointer for streaming raw UMIs
    h5_raw = h5py.File(INPUT_FILE, "r")
    umi_grp = h5_raw["layers"]["UMIs"]
    umi_indptr = umi_grp["indptr"]  # Index pointers for CSR rows

    #load relevant genes into memory in chunks, deleting each object as you go.
    chunks = []
    start_time = time.time()

    for start_idx in range(0, total_cells, CHUNK_SIZE):
        end_idx = min(start_idx + CHUNK_SIZE, total_cells)

        pct = (end_idx / total_cells) * 100
        print(
            f"[{pct:5.1f}%] Processed: {end_idx:>8,} / {total_cells:,}  |"
            f"  {total_cells - end_idx:>8,} cells to go",
            end="\r",
            flush=True,
        )

        #Pull 10k rows of X from disk & slice to 5 genes
        batch = adata_backed[start_idx:end_idx, :].to_memory()
        batch_5genes = batch[:, valid_genes].copy()

        #Pull matching 10k rows from layers['UMIs'] via CSR pointer
        p0 = int(umi_indptr[start_idx])
        p1 = int(umi_indptr[end_idx])

        data_chunk = umi_grp["data"][p0:p1]
        indices_chunk = umi_grp["indices"][p0:p1]
        indptr_chunk = umi_indptr[start_idx : end_idx + 1] - p0

        #Reconstruct 10k sparse matrix and slice to target genes
        umi_csr = sp.csr_matrix(
            (data_chunk, indices_chunk, indptr_chunk),
            shape=(end_idx - start_idx, adata_backed.n_vars),
        )
        batch_5genes.layers["UMIs"] = umi_csr[:, gene_indices].copy()

        # Drop heavy cell-cell pairwise graphs, but KEEP obsm (X_scvi, X_umap) for UMAP plotting.
        batch_5genes.obsp.clear()

        chunks.append(batch_5genes)

        del batch, batch_5genes, umi_csr
        gc.collect()

    elapsed = time.time() - start_time
    print(f"\n\nExtraction complete in {elapsed:.1f} seconds.")

    # Safely close backed pointer via del
    del adata_backed
    gc.collect()


    # Merging chunks and saving.
    print("Concatenating batches into final AnnData...")
    final_adata = ad.concat(chunks)
    del chunks
    gc.collect()

    # Replace any '/' in metadata columns so HDF5 doesn't treat them as folder paths
    final_adata.obs.columns = [c.replace("/", "_") for c in final_adata.obs.columns]
    final_adata.var.columns = [c.replace("/", "_") for c in final_adata.var.columns]

    # assign microglia, astrocyte, neuronal categories.
    assign_APJ_Cats(final_adata)

    #Print summary breakdown
    print("Cell counts per category in 'APJ_Cats':")
    print(final_adata.obs['APJ_Cats'].value_counts())

    print(f"Writing to {OUTPUT_FILE} ...")
    final_adata.write_h5ad(OUTPUT_FILE)

    print("\n" + "=" * 60)
    print(f"SUCCESS: Saved {OUTPUT_FILE}")
    print(f"Total cells kept: {final_adata.n_obs:,} (100% of cells)")
    print(f"Total genes kept: {final_adata.n_vars} {list(final_adata.var_names)}")
    print("=" * 60)